<a href="https://colab.research.google.com/github/D-Roumaissa/Customer-Retention-and-Churn-Behavioral-Analysis./blob/main/Customer_Retention_and_Churn_Behavioral_Analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [8]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import gradio as gr
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder

# Load and Clean
url = "https://raw.githubusercontent.com/IBM/telco-customer-churn-on-icp4d/master/data/Telco-Customer-Churn.csv"
df = pd.read_csv(url)
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce').fillna(0)

# Targeted Analysis: Generating the 3 Management Reports
def create_management_reports(df):
    sns.set_style("white")

    # 1. Bar Chart (Contract Report)
    plt.figure(figsize=(8, 5))
    sns.barplot(x='Contract', y=(df['Churn'] == 'Yes').astype(int), data=df,
                palette=['#e74c3c', '#2ecc71', '#3498db'], errorbar=None)
    plt.title('Churn Risk by Membership Type', fontsize=14, fontweight='bold')
    plt.ylabel('Churn Probability')
    plt.savefig('contract_report.png')
    plt.close()

    # 2. Circle/Donut Chart (Portions)
    plt.figure(figsize=(6, 6))
    counts = df['Contract'].value_counts()
    plt.pie(counts, labels=counts.index, autopct='%1.1f%%', startangle=140,
            colors=['#2c3e50', '#2980b9', '#95a5a6'], wedgeprops={'width': 0.4})
    plt.title("Membership Distribution", fontsize=14, fontweight='bold')
    plt.savefig('donut_report.png', transparent=True)
    plt.close()

    # 3. Curve Graph (Behavioral Lifecycle)
    plt.figure(figsize=(10, 4))
    sns.kdeplot(df[df['Churn']=='No']['tenure'], fill=True, color="#27ae60", label="Retained")
    sns.kdeplot(df[df['Churn']=='Yes']['tenure'], fill=True, color="#c0392b", label="Churned")
    plt.title("Retention vs Churn Curves", fontsize=14, fontweight='bold')
    plt.xlabel("Months of Membership")
    plt.legend()
    plt.savefig('curve_report.png')
    plt.close()

create_management_reports(df)

/tmp/ipykernel_66198/2103830463.py:20: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.barplot(x='Contract', y=(df['Churn'] == 'Yes').astype(int), data=df,


In [9]:
# Encoding for the model
le = LabelEncoder()
df_model = df.copy()
for col in ['Contract', 'InternetService', 'OnlineSecurity', 'TechSupport']:
    df_model[col] = le.fit_transform(df_model[col])

X = df_model[['Contract', 'tenure', 'MonthlyCharges', 'SeniorCitizen']]
y = (df_model['Churn'] == 'Yes').astype(int)

clf = RandomForestClassifier(n_estimators=100, max_depth=5, random_state=42)
clf.fit(X, y)

RandomForestClassifier(max_depth=5, random_state=42)

In [10]:
def analyze_subscriber(contract_type, monthly_spend, months_active):
    contract_map = {"Month-to-month": 0, "One year": 1, "Two year": 2}
    features = np.array([[contract_map[contract_type], months_active, monthly_spend, 0]])
    prob = clf.predict_proba(features)[0][1]

    if contract_type == "Month-to-month" and prob > 0.4:
        advice = "⚠️ HIGH ALERT: Monthly subscribers show 4x higher churn. Suggest conversion to Yearly."
    elif prob < 0.2:
        advice = "✅ STABLE: High retention probability. Ideal for loyalty rewards."
    else:
        advice = "ℹ️ MODERATE: Monitor usage patterns closely."

    return f"{prob:.1%}", advice

with gr.Blocks(theme=gr.themes.Default(primary_hue="blue", secondary_hue="slate")) as demo:
    gr.Markdown("# 🏦 Customer-Retention-and-Churn-Behavioral-Analysis")
    gr.Markdown("### Strategic Financial Dashboard: Membership Attrition Analytics")

    with gr.Row():
        # Input Column
        with gr.Column(scale=1):
            gr.Markdown("### 👤 Client Data Input")
            contract = gr.Radio(["Month-to-month", "One year", "Two year"], label="Membership Type", value="Month-to-month")
            spend = gr.Number(label="Monthly Subscription Fee ($)", value=70.0)
            tenure = gr.Slider(1, 72, label="Months as a Member", value=5)
            btn = gr.Button("Analyze Behavior", variant="primary")

            gr.Markdown("### 📈 Risk Result")
            result_prob = gr.Label(label="Churn Probability")
            result_txt = gr.Textbox(label="Management Recommendation")

        # Visual Analytics Column
        with gr.Column(scale=2):
            gr.Markdown("### 📊 Business Intelligence Reports")
            with gr.Row():
                gr.Image("donut_report.png", label="Membership Portions")
                gr.Image("contract_report.png", label="Risk by Contract")
            gr.Image("curve_report.png", label="Behavioral Retention Curves")

    btn.click(fn=analyze_subscriber, inputs=[contract, spend, tenure], outputs=[result_prob, result_txt])

demo.launch()

/tmp/ipykernel_66198/871699075.py:15: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(theme=gr.themes.Default(primary_hue="blue", secondary_hue="slate")) as demo:


It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://2f585d1f079622f552.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
